# Create time delta from dates
Useful for survival analysis (a prerequisit for the Kaplan Meier)

In [1]:
import base64
import json
import requests

In [2]:
with open("token.txt", "r") as f:
    token = f.read().strip()
headers = {
    "Authorization": token
}

In [3]:
# Set the collaboration ID:
#
#   - 2: Test collaboration (with IKNL and UPM)
#   - 3: IDEA4RC collaboration
#
COLLABORATION_ID = 2

In [4]:
# Set the organization IDs that are part of this workspace. These should be the IDs of
# the vantage6 organizations:
#
#   1	- root
#   2	- ENG
#   3	- UPM
#   4	- INT
#   5	- UKE
#   6	- CLB
#   7	- FPNS
#
# These are basically all organization that are part of the workspace (thus the same
# list as in the `1-new-workspace.ipynb` notebook):
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/organization?collaboration_id={COLLABORATION_ID}",
    headers=headers
)
ORGANIZATION_IDS = [org["id"] for org in response.json()["data"]]
ORGANIZATION_IDS

[1, 3]

In [5]:
# Set the study ID. This is the `study` id that belongs to the RAVEN workspace. See the
# `0-new-workspace.ipynb` notebook for more information.
STUDY_ID = 4
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
# See the `1-new-analysis.ipynb` notebook for more information.
SESSION_ID = 3

In [6]:
# The image to use is the latest version of the analytics algorithm
IMAGE = "harbor2.vantage6.ai/idea4rc/preprocessing:latest"
#
# The method (that is within this IMAGE) to execute is the summary algorithm. For
# data exploration we use the `summary` method.
METHOD = "timedelta"

In [7]:
# Set the study ID. This is the `study` id that belongs to the RAVEN workspace. See the
# `0-new-workspace.ipynb` notebook for more information.
STUDY_ID = 4
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
# See the `1-new-analysis.ipynb` notebook for more information.
SESSION_ID = 3

In [8]:
# In the idea4rc case, preprocessing should always be applied to all cohorts (= vantage6
# dataframes). This because we always want all datasets to have the same columns and
# types.
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/{SESSION_ID}/dataframe?per_page=999",
    headers=headers
)
DATAFRAME_IDS = [df["id"] for df in response.json()["data"]]
DATAFRAME_IDS

[99, 100]

## REPEAT FOR EACH DATAFRAME

In [ ]:
def payload(df_id):
    return {
        "dataframe_id": df_id,
        "task": {
            "image": IMAGE,
        "method": METHOD,
        "organizations": [
            {
                "id": org_id,
                "arguments": base64.b64encode(
                    json.dumps(
                        {
                            "column": "histology",
                            "output_column": "histology_merged",
                            "mapping": {
                                "Liposarcoma": [
                                    "1004/1007 liposarcoma",
                                    "1010 leiomyosarcoma",
                                    "1013 solitary fibrous tumour",
                                    "1016 mpnst",
                                ],
                                "Other sarcomas": [
                                    "1022 other sarcomas"
                                ]
                            }
                        }
                    ).encode("UTF-8")
                ).decode("UTF-8")
            } for org_id in ORGANIZATION_IDS # we need to repeat this for each org
        ],
    }
}

In [ ]:
for df_id in DATAFRAME_IDS: 
    response = requests.post(
        f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/dataframe/{df_id}/preprocess",
        headers=headers,
        json=payload(df_id)
    )
    print(response.json())
# In the response we need to extract the task ID and the job ID so we can poll werther
# the (central) task is finished. Later on we can also use these IDs to retrieve the
# results.
# response.json()

{'session': {'id': 3, 'link': '/server/session/3', 'methods': ['DELETE', 'PATCH', 'GET']}, 'tasks': '/server/task?dataframe_id=99', 'last_session_task': {'status': 'awaiting', 'finished_at': None, 'action': 'preprocessing', 'collaboration': {'id': 2, 'link': '/server/collaboration/2', 'methods': ['DELETE', 'PATCH', 'GET']}, 'runs': '/server/run?task_id=506', 'results': '/server/result?task_id=506', 'parent': None, 'depends_on': [465], 'required_by': [], 'children': '/server/task?parent_id=506', 'init_org': {'id': 1, 'link': '/server/organization/1', 'methods': ['DELETE', 'PATCH', 'GET']}, 'init_user': {'id': 1, 'link': '/server/user/1', 'methods': ['DELETE', 'PATCH', 'GET']}, 'databases': [{'label': None, 'type': 'dataframe', 'dataframe_id': 99, 'dataframe_name': 'gifted_bassi', 'position': 0}], 'study': {'id': 4, 'link': '/server/study/4', 'methods': ['DELETE', 'PATCH', 'GET']}, 'algorithm_store': None, 'session': {'id': 3, 'link': '/server/session/3', 'methods': ['DELETE', 'PATCH', '